# Statistical Analysis of PM2.5 Concentrations Across Five Indian Cities

## Objective

This notebook extends the exploratory data analysis by statistically evaluating the spatial, seasonal, meteorological, and pollutant-related factors associated with daily PM2.5 concentrations across Bengaluru, Chennai, Delhi, Kolkata, and Mumbai during 2020–2021.

The analysis aims to determine whether the patterns observed during EDA are statistically supported and to identify variables that may be useful for subsequent predictive modeling.

### Main Questions

1. Do PM2.5 concentrations differ significantly across cities?
2. Do PM2.5 concentrations differ significantly across seasons?
3. How are meteorological conditions associated with PM2.5 concentrations?
4. How strongly are other air pollutants associated with PM2.5?
5. Which factors remain important when multiple predictors are considered simultaneously?

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv(
    "air_pollution_weather_cleaned_2020_2021 (1).csv"
)

df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (3655, 13)

Date range:
2020-01-01 00:00:00 to 2021-12-31 00:00:00

Columns:
['Date', 'City', 'PM2.5', 'PM10', 'NO2', 'SO2', 'O3', 'CO', 'Temperature_Mean', 'Relative_Humidity_Mean', 'Precipitation', 'Wind_Speed_Max', 'Surface_Pressure_Mean']


,Date,City,PM2.5,PM10,NO2,SO2,O3,CO,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean
0,2020-01-01,Bengaluru,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587,21.6,80,1.9,18.5,915.0
1,2020-01-02,Bengaluru,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079,22.4,77,1.0,15.1,916.0
2,2020-01-03,Bengaluru,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273,23.0,73,0.0,14.0,915.0
3,2020-01-04,Bengaluru,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189,23.4,70,0.0,13.2,913.6
4,2020-01-05,Bengaluru,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578,23.3,73,0.2,16.6,913.4


## 1. Statistical Comparison of PM2.5 Across Cities

The exploratory analysis indicated substantial differences in PM2.5 concentrations among the five cities. Statistical hypothesis testing is used to determine whether these observed differences are statistically significant.

### Hypotheses

**Null hypothesis (H₀):** The distribution of daily PM2.5 concentrations is the same across the five cities.

**Alternative hypothesis (H₁):** At least one city has a different distribution of daily PM2.5 concentrations.

In [ ]:
# Extract non-missing PM2.5 observations

pm25_test_data = df[
    ["City", "PM2.5"]
].dropna()

# Shapiro-Wilk normality test separately for each city

normality_results = []

for city in pm25_test_data["City"].unique():

    values = pm25_test_data.loc[
        pm25_test_data["City"] == city,
        "PM2.5"
    ]

    statistic, p_value = stats.shapiro(values)

    normality_results.append({
        "City": city,
        "Observations": len(values),
        "Shapiro_W": statistic,
        "P_Value": p_value
    })

normality_results = pd.DataFrame(normality_results)

display(normality_results.round(4))

,City,Observations,Shapiro_W,P_Value
0,Bengaluru,638,0.9108,0.0
1,Chennai,726,0.9024,0.0
2,Delhi,726,0.8306,0.0
3,Kolkata,688,0.8649,0.0
4,Mumbai,725,0.4735,0.0


### Normality Assessment

The Shapiro–Wilk test produced statistically significant results (p < 0.001) for all five cities, indicating that daily PM2.5 concentrations deviate significantly from a normal distribution.

This finding is consistent with the exploratory analysis, which showed right-skewed distributions and several high-pollution observations. Therefore, a non-parametric Kruskal–Wallis test is used to compare PM2.5 concentrations across cities.

In [ ]:
# Create PM2.5 groups for each city

city_groups = [
    group["PM2.5"].dropna().values
    for _, group in df.groupby("City")
]

# Kruskal-Wallis test

kw_statistic, kw_pvalue = stats.kruskal(*city_groups)

print("Kruskal-Wallis Test: PM2.5 Across Cities")
print("-----------------------------------------")
print(f"H statistic: {kw_statistic:.4f}")
print(f"P-value: {kw_pvalue:.6e}")

if kw_pvalue < 0.05:
    print("\nResult: Reject the null hypothesis.")
    print("PM2.5 distributions differ significantly across cities.")
else:
    print("\nResult: Fail to reject the null hypothesis.")

Kruskal-Wallis Test: PM2.5 Across Cities
-----------------------------------------
H statistic: 709.5100
P-value: 3.040894e-152

Result: Reject the null hypothesis.
PM2.5 distributions differ significantly across cities.


### Kruskal–Wallis Test Result

The Kruskal–Wallis test indicated a statistically significant difference in daily PM2.5 distributions across the five cities (H = 709.51, p < 0.001).

Therefore, the null hypothesis is rejected, providing strong evidence that PM2.5 concentrations differ across cities. However, the Kruskal–Wallis test does not identify which specific city pairs differ significantly. Pairwise post-hoc comparisons are therefore required.

In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests

cities = sorted(pm25_test_data["City"].unique())

pairwise_results = []

for city1, city2 in combinations(cities, 2):

    group1 = pm25_test_data.loc[
        pm25_test_data["City"] == city1,
        "PM2.5"
    ]

    group2 = pm25_test_data.loc[
        pm25_test_data["City"] == city2,
        "PM2.5"
    ]

    statistic, p_value = stats.mannwhitneyu(
        group1,
        group2,
        alternative="two-sided"
    )

    pairwise_results.append({
        "City_1": city1,
        "City_2": city2,
        "U_Statistic": statistic,
        "Raw_P_Value": p_value
    })

pairwise_results = pd.DataFrame(pairwise_results)

# Holm correction for multiple comparisons
reject, adjusted_p, _, _ = multipletests(
    pairwise_results["Raw_P_Value"],
    alpha=0.05,
    method="holm"
)

pairwise_results["Adjusted_P_Value"] = adjusted_p
pairwise_results["Significant"] = reject

display(
    pairwise_results.round({
        "U_Statistic": 2,
        "Raw_P_Value": 6,
        "Adjusted_P_Value": 6
    })
)

,City_1,City_2,U_Statistic,Raw_P_Value,Adjusted_P_Value,Significant
0,Bengaluru,Chennai,244357.0,0.078704,0.078704,False
1,Bengaluru,Delhi,63481.0,0.000000,0.000000,True
2,Bengaluru,Kolkata,172429.5,0.000000,0.000000,True
3,Bengaluru,Mumbai,186989.5,0.000000,0.000000,True
4,Chennai,Delhi,69917.0,0.000000,0.000000,True
5,Chennai,Kolkata,186450.0,0.000000,0.000000,True
6,Chennai,Mumbai,202798.0,0.000000,0.000000,True
7,Delhi,Kolkata,351989.5,0.000000,0.000000,True
8,Delhi,Mumbai,388074.5,0.000000,0.000000,True
9,Kolkata,Mumbai,265420.0,0.036659,0.073317,False


### Post-hoc Pairwise Comparisons

Pairwise Mann–Whitney U tests with Holm correction were conducted following the significant Kruskal–Wallis test.

Eight of the ten city-pair comparisons showed statistically significant differences in PM2.5 distributions after adjustment for multiple comparisons.

No statistically significant difference was detected between Bengaluru and Chennai (adjusted p = 0.079) or between Kolkata and Mumbai (adjusted p = 0.073). All other city pairs differed significantly.

These results indicate substantial spatial variation in PM2.5 concentrations, while also suggesting similarities between Bengaluru and Chennai and between Kolkata and Mumbai during the study period.

## 2. Statistical Comparison of PM2.5 Across Seasons

Exploratory analysis revealed substantial seasonal variation in PM2.5 concentrations, with generally higher concentrations during winter and post-monsoon periods and lower concentrations during the monsoon season.

A Kruskal–Wallis test is used to determine whether PM2.5 distributions differ significantly across seasons.

### Hypotheses

**Null hypothesis (H₀):** The distribution of PM2.5 concentrations is the same across all seasons.

**Alternative hypothesis (H₁):** At least one season has a different PM2.5 distribution.

In [ ]:
# Define seasons

def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-Monsoon"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"


df["Season"] = df["Date"].dt.month.apply(assign_season)

season_order = [
    "Winter",
    "Pre-Monsoon",
    "Monsoon",
    "Post-Monsoon"
]

# Create groups
season_groups = [
    df.loc[
        (df["Season"] == season) & df["PM2.5"].notna(),
        "PM2.5"
    ].values
    for season in season_order
]

# Kruskal-Wallis test
season_kw_stat, season_kw_p = stats.kruskal(
    *season_groups
)

print("Kruskal-Wallis Test: PM2.5 Across Seasons")
print("------------------------------------------")
print(f"H statistic: {season_kw_stat:.4f}")
print(f"P-value: {season_kw_p:.6e}")

if season_kw_p < 0.05:
    print("\nResult: Reject the null hypothesis.")
    print("PM2.5 distributions differ significantly across seasons.")
else:
    print("\nResult: Fail to reject the null hypothesis.")

Kruskal-Wallis Test: PM2.5 Across Seasons
------------------------------------------
H statistic: 1345.1844
P-value: 2.309802e-291

Result: Reject the null hypothesis.
PM2.5 distributions differ significantly across seasons.


### Seasonal Kruskal–Wallis Test Result

The Kruskal–Wallis test revealed a statistically significant difference in PM2.5 distributions across seasons (H = 1345.18, p < 0.001).

Therefore, the null hypothesis is rejected, indicating strong seasonal variation in PM2.5 concentrations. Post-hoc pairwise comparisons are conducted to determine which seasons differ significantly.

In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests

season_pairwise = []

for season1, season2 in combinations(season_order, 2):

    group1 = df.loc[
        (df["Season"] == season1) & df["PM2.5"].notna(),
        "PM2.5"
    ]

    group2 = df.loc[
        (df["Season"] == season2) & df["PM2.5"].notna(),
        "PM2.5"
    ]

    statistic, p_value = stats.mannwhitneyu(
        group1,
        group2,
        alternative="two-sided"
    )

    season_pairwise.append({
        "Season_1": season1,
        "Season_2": season2,
        "U_Statistic": statistic,
        "Raw_P_Value": p_value
    })

season_pairwise = pd.DataFrame(season_pairwise)

# Holm correction
reject, adjusted_p, _, _ = multipletests(
    season_pairwise["Raw_P_Value"],
    alpha=0.05,
    method="holm"
)

season_pairwise["Adjusted_P_Value"] = adjusted_p
season_pairwise["Significant"] = reject

display(
    season_pairwise.round({
        "U_Statistic": 2,
        "Raw_P_Value": 6,
        "Adjusted_P_Value": 6
    })
)

,Season_1,Season_2,U_Statistic,Raw_P_Value,Adjusted_P_Value,Significant
0,Winter,Pre-Monsoon,630023.5,0.0,0.0,True
1,Winter,Monsoon,951990.5,0.0,0.0,True
2,Winter,Post-Monsoon,341447.0,0.0,0.0,True
3,Pre-Monsoon,Monsoon,682543.5,0.0,0.0,True
4,Pre-Monsoon,Post-Monsoon,171001.5,0.0,0.0,True
5,Monsoon,Post-Monsoon,117342.0,0.0,0.0,True


### Post-hoc Seasonal Comparison Results

Pairwise Mann–Whitney U tests with Holm correction showed statistically significant differences in PM2.5 distributions for all six seasonal comparisons (adjusted p < 0.001).

Combined with the exploratory seasonal analysis, these results provide strong evidence of a systematic seasonal pattern in PM2.5 concentrations. Pollution levels were generally highest during winter and post-monsoon periods and lowest during the monsoon season, although the magnitude of this seasonal variation differed across cities.

## 3. Association Between Meteorological Factors and PM2.5

Exploratory analysis suggested that meteorological conditions may influence daily PM2.5 concentrations. Since PM2.5 is strongly non-normally distributed and relationships may not be strictly linear, Spearman rank correlation is used to quantify the association between PM2.5 and meteorological variables.

The analysis considers mean temperature, relative humidity, precipitation, maximum wind speed, and mean surface pressure.

In [ ]:
weather_variables = [
    "Temperature_Mean",
    "Relative_Humidity_Mean",
    "Precipitation",
    "Wind_Speed_Max",
    "Surface_Pressure_Mean"
]

spearman_results = []

for variable in weather_variables:

    temp = df[["PM2.5", variable]].dropna()

    rho, p_value = stats.spearmanr(
        temp["PM2.5"],
        temp[variable]
    )

    spearman_results.append({
        "Variable": variable,
        "Spearman_Rho": rho,
        "P_Value": p_value
    })

spearman_results = pd.DataFrame(spearman_results)

# Sort by strength of association
spearman_results["Absolute_Rho"] = (
    spearman_results["Spearman_Rho"].abs()
)

spearman_results = (
    spearman_results
    .sort_values("Absolute_Rho", ascending=False)
    .drop(columns="Absolute_Rho")
    .reset_index(drop=True)
)

display(spearman_results.round(4))

,Variable,Spearman_Rho,P_Value
0,Precipitation,-0.6021,0.0
1,Relative_Humidity_Mean,-0.5250,0.0
2,Temperature_Mean,-0.4025,0.0
3,Wind_Speed_Max,-0.3820,0.0
4,Surface_Pressure_Mean,0.2036,0.0


### Spearman Correlation Results

Spearman rank correlation revealed statistically significant associations between PM2.5 concentrations and all five meteorological variables (p < 0.001).

Precipitation showed the strongest negative association with PM2.5 (ρ = -0.602), followed by relative humidity (ρ = -0.525), mean temperature (ρ = -0.403), and maximum wind speed (ρ = -0.382). Mean surface pressure showed a weaker positive association (ρ = 0.204).

The stronger Spearman association observed for precipitation compared with the earlier Pearson correlation suggests that some meteorological relationships with PM2.5 may be non-linear or influenced by extreme observations. These bivariate associations should not be interpreted as independent causal effects because meteorological conditions, season, city, and other pollutants may be interrelated.

## 4. Multivariable Analysis of Factors Associated with PM2.5

Bivariate correlations do not account for relationships among predictors or differences between cities and seasons. Therefore, a multiple regression model is used to examine the simultaneous association of meteorological conditions, co-pollutants, spatial variation, seasonal variation, and temporal trend with daily PM2.5 concentrations.

Because PM2.5 concentrations are strongly right-skewed, the response variable is transformed using log(1 + PM2.5) before modelling.

In [ ]:
import numpy as np

regression_columns = [
    "Date", "City", "Season",
    "PM2.5", "PM10", "NO2", "SO2", "O3", "CO",
    "Temperature_Mean",
    "Relative_Humidity_Mean",
    "Precipitation",
    "Wind_Speed_Max",
    "Surface_Pressure_Mean"
]

regression_data = (
    df[regression_columns]
    .dropna()
    .copy()
)

# Log-transform PM2.5
regression_data["Log_PM25"] = np.log1p(
    regression_data["PM2.5"]
)

# Add a numerical time trend
regression_data["Time_Index"] = (
    regression_data["Date"] -
    regression_data["Date"].min()
).dt.days

print("Regression dataset shape:")
print(regression_data.shape)

print("\nObservations removed due to missing values:")
print(len(df) - len(regression_data))

print("\nPM2.5 skewness before transformation:")
print(round(regression_data["PM2.5"].skew(), 3))

print("\nPM2.5 skewness after log transformation:")
print(round(regression_data["Log_PM25"].skew(), 3))

display(regression_data.head())

Regression dataset shape:
(3323, 16)

Observations removed due to missing values:
332

PM2.5 skewness before transformation:
4.089

PM2.5 skewness after log transformation:
0.313


,Date,City,Season,PM2.5,PM10,NO2,SO2,O3,CO,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean,Log_PM25,Time_Index
0,2020-01-01,Bengaluru,Winter,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587,21.6,80,1.9,18.5,915.0,3.321545,0
1,2020-01-02,Bengaluru,Winter,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079,22.4,77,1.0,15.1,916.0,3.231423,1
2,2020-01-03,Bengaluru,Winter,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273,23.0,73,0.0,14.0,915.0,3.423131,2
3,2020-01-04,Bengaluru,Winter,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189,23.4,70,0.0,13.2,913.6,4.029553,3
4,2020-01-05,Bengaluru,Winter,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578,23.3,73,0.2,16.6,913.4,3.956885,4


### Multicollinearity Assessment

Before fitting the multivariable regression model, multicollinearity among the continuous predictors is assessed using the Variance Inflation Factor (VIF).

High multicollinearity can make regression coefficients unstable and complicate the interpretation of individual predictor effects.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

vif_variables = [
    "PM10",
    "NO2",
    "SO2",
    "O3",
    "CO",
    "Temperature_Mean",
    "Relative_Humidity_Mean",
    "Precipitation",
    "Wind_Speed_Max",
    "Surface_Pressure_Mean",
    "Time_Index"
]

X_vif = regression_data[vif_variables].copy()

# Add intercept
X_vif = sm.add_constant(X_vif)

vif_results = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

# Remove intercept from displayed results
vif_results = (
    vif_results[vif_results["Variable"] != "const"]
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

display(vif_results.round(2))

,Variable,VIF
0,PM10,4.57
1,NO2,3.53
2,Relative_Humidity_Mean,2.62
3,Temperature_Mean,2.22
4,CO,2.18
5,SO2,1.60
6,Surface_Pressure_Mean,1.46
7,Wind_Speed_Max,1.37
8,Precipitation,1.32
9,O3,1.28


### Multicollinearity Assessment Results

The VIF analysis indicated no severe multicollinearity among the continuous predictors. All VIF values were below 5, with PM10 showing the highest value (VIF = 4.57), followed by NO2 (VIF = 3.53).

Therefore, all selected predictors were retained for the multivariable regression analysis.

In [ ]:
import statsmodels.formula.api as smf

regression_model = smf.ols(
    formula="""
    Log_PM25 ~ PM10 + NO2 + SO2 + O3 + CO
    + Temperature_Mean
    + Relative_Humidity_Mean
    + Precipitation
    + Wind_Speed_Max
    + Surface_Pressure_Mean
    + C(City)
    + C(Season)
    + Time_Index
    """,
    data=regression_data
).fit()

print(regression_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Log_PM25   R-squared:                       0.828
Model:                            OLS   Adj. R-squared:                  0.827
Method:                 Least Squares   F-statistic:                     880.9
Date:                Mon, 20 Jul 2026   Prob (F-statistic):               0.00
Time:                        13:00:15   Log-Likelihood:                -1240.3
No. Observations:                3323   AIC:                             2519.
Df Residuals:                    3304   BIC:                             2635.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

### Initial Multivariable Regression Results

The multiple linear regression model explained approximately 82.8% of the variation in log-transformed PM2.5 concentrations (R² = 0.828; adjusted R² = 0.827).

Several meteorological, pollutant, spatial, seasonal, and temporal variables showed statistically significant adjusted associations with PM2.5. However, diagnostic statistics indicated substantial positive residual autocorrelation (Durbin–Watson = 0.783) and non-normal, heavy-tailed residuals.

Therefore, the conventional OLS standard errors and significance tests should be interpreted cautiously. Additional diagnostic assessment and robust inference are required before drawing final conclusions from the regression coefficients.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Breusch-Pagan test for heteroskedasticity

bp_test = het_breuschpagan(
    regression_model.resid,
    regression_model.model.exog
)

bp_results = {
    "LM Statistic": bp_test[0],
    "LM Test P-value": bp_test[1],
    "F Statistic": bp_test[2],
    "F Test P-value": bp_test[3]
}

print("Breusch-Pagan Test for Heteroskedasticity")
print("------------------------------------------")

for key, value in bp_results.items():
    print(f"{key}: {value:.6e}")

Breusch-Pagan Test for Heteroskedasticity
------------------------------------------
LM Statistic: 1.271669e+02
LM Test P-value: 1.839119e-18
F Statistic: 7.303945e+00
F Test P-value: 7.427921e-19


### Regression Diagnostic Assessment

The Breusch–Pagan test indicated significant heteroskedasticity (p < 0.001), showing that the variance of the regression residuals is not constant.

Together with the previously observed positive residual autocorrelation (Durbin–Watson = 0.783), this indicates violations of the classical OLS assumptions underlying conventional standard errors and significance tests.

Therefore, robust inference is required before interpreting the statistical significance of the regression coefficients.

In [ ]:
# Obtain HAC (Newey-West) robust standard errors
# maxlags=7 allows residual dependence across approximately one week

regression_hac = regression_model.get_robustcov_results(
    cov_type="HAC",
    maxlags=7
)

print(regression_hac.summary())

                            OLS Regression Results                            
Dep. Variable:               Log_PM25   R-squared:                       0.828
Model:                            OLS   Adj. R-squared:                  0.827
Method:                 Least Squares   F-statistic:                     299.8
Date:                Mon, 20 Jul 2026   Prob (F-statistic):               0.00
Time:                        13:05:39   Log-Likelihood:                -1240.3
No. Observations:                3323   AIC:                             2519.
Df Residuals:                    3304   BIC:                             2635.
Df Model:                          18                                         
Covariance Type:                  HAC                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

### HAC-Robust Regression Results

To account for heteroskedasticity and residual autocorrelation, the regression model was evaluated using heteroskedasticity and autocorrelation consistent (HAC/Newey–West) standard errors with seven lags.

The model explained approximately 82.8% of the variation in log-transformed PM2.5 concentrations (R² = 0.828).

After robust adjustment, PM10 showed a strong positive association with PM2.5, while relative humidity, precipitation, and maximum wind speed showed significant negative associations. Surface pressure showed a significant positive association, and O3 retained a weaker positive association.

Temperature, NO2, SO2, CO, and the overall linear time trend were not statistically significant after robust adjustment, despite some appearing significant under conventional OLS inference.

Seasonal effects remained important. Compared with the reference monsoon season, winter and post-monsoon periods were associated with significantly higher PM2.5 levels after controlling for meteorological conditions, co-pollutants, city, and temporal trend.

These findings represent adjusted statistical associations and should not be interpreted as causal effects.

In [ ]:
# Extract HAC coefficients, confidence intervals and p-values

hac_results = pd.DataFrame({
    "Variable": regression_model.params.index,
    "Coefficient": regression_hac.params,
    "P_Value": regression_hac.pvalues,
    "CI_Lower": regression_hac.conf_int()[:, 0],
    "CI_Upper": regression_hac.conf_int()[:, 1]
})

# Convert log-scale coefficients to percentage change
hac_results["Percent_Change"] = (
    np.exp(hac_results["Coefficient"]) - 1
) * 100

hac_results["CI_Lower_Percent"] = (
    np.exp(hac_results["CI_Lower"]) - 1
) * 100

hac_results["CI_Upper_Percent"] = (
    np.exp(hac_results["CI_Upper"]) - 1
) * 100

display(
    hac_results[
        [
            "Variable",
            "Coefficient",
            "Percent_Change",
            "P_Value",
            "CI_Lower_Percent",
            "CI_Upper_Percent"
        ]
    ].round(3)
)

,Variable,Coefficient,Percent_Change,P_Value,CI_Lower_Percent,CI_Upper_Percent
0,Intercept,-6.198,-99.797,0.140,-100.000,659.875
1,C(City)[T.Chennai],-0.961,-61.765,0.032,-84.092,-8.103
2,C(City)[T.Delhi],-0.545,-42.016,0.101,-69.799,11.328
3,C(City)[T.Kolkata],-0.902,-59.429,0.041,-82.892,-3.790
4,C(City)[T.Mumbai],-0.845,-57.029,0.060,-82.194,3.705
5,C(Season)[T.Post-Monsoon],0.289,33.525,0.000,23.679,44.154
6,C(Season)[T.Pre-Monsoon],0.048,4.936,0.293,-4.084,14.804
7,C(Season)[T.Winter],0.421,52.413,0.000,39.157,66.931
8,PM10,0.005,0.533,0.000,0.454,0.612
9,NO2,0.001,0.118,0.281,-0.097,0.334


### Interpretation of Adjusted Effect Estimates

Transformation of the log-scale regression coefficients provides a more interpretable representation of the adjusted associations.

Seasonal effects were substantial. Holding the other model variables constant, winter was associated with approximately 52% higher PM2.5+1 levels than the monsoon season, while the post-monsoon period was associated with approximately 34% higher levels. The pre-monsoon difference was not statistically significant.

Among meteorological variables, maximum wind speed showed one of the clearest negative adjusted associations: a one-unit increase in maximum wind speed was associated with approximately a 1.8% decrease in PM2.5+1. Higher relative humidity and precipitation were also associated with lower PM2.5 levels, whereas higher surface pressure showed a positive association.

Among co-pollutants, PM10 showed the strongest positive adjusted association with PM2.5. O3 showed a comparatively small positive association, while NO2, SO2, and CO were not statistically significant after robust adjustment.

These estimates represent conditional statistical associations rather than causal effects.

## 5. Predictive Modelling of Daily PM2.5 Concentrations

The inferential analysis identified significant spatial, seasonal, meteorological, and co-pollutant associations with PM2.5 concentrations. The next stage evaluates whether these variables can be used to predict daily PM2.5 concentrations.

Because the observations form a chronological daily dataset, model evaluation is performed using a time-aware approach rather than a random train-test split in order to reduce temporal data leakage.

## 6. Summary of Statistical Analysis

The statistical analysis confirmed substantial spatial and seasonal variation in PM2.5 concentrations across the five study cities.

PM2.5 distributions differed significantly across cities, although Bengaluru and Chennai and Kolkata and Mumbai did not show statistically significant pairwise differences after multiple-comparison adjustment. Significant differences were observed across all seasonal comparisons, with particularly elevated concentrations during winter and post-monsoon periods.

Spearman correlation analysis identified significant associations between PM2.5 and meteorological conditions. Precipitation, relative humidity, temperature, and wind speed showed negative associations, while surface pressure showed a weaker positive association.

Multivariable regression was performed using log-transformed PM2.5 while controlling for meteorological variables, co-pollutants, city, season, and temporal trend. Multicollinearity was not severe based on VIF values. Diagnostic testing identified heteroskedasticity and positive residual autocorrelation, so HAC-robust standard errors were used for statistical inference.

After robust adjustment, PM10 remained strongly positively associated with PM2.5, while relative humidity, precipitation, and wind speed showed significant negative associations. Winter and post-monsoon periods remained important after adjustment for other variables.

These findings support the use of pollutant, meteorological, temporal, and city-related information in the next stage of the project: forecasting next-day PM2.5 concentrations.

The subsequent feature-engineering stage will construct the next-day PM2.5 target, lagged pollution variables, rolling historical features, and temporal predictors while ensuring that no future information is introduced into the forecasting models.